# Conditional Card 29 Counterfactual

## Objective

The panel-level analysis identified Card 29 as the strongest intervention candidate:

- it receives the highest click volume;
- it converts clicks into sales materially worse than the panel;
- it attracts substantially more clicks than key alternatives when offered in the same application.

A complete removal of Card 29 is not necessarily the most practical intervention.

If Card 29 is the customer's only available offer, it cannot divert traffic away from another card. This analysis therefore evaluates a more realistic policy:

> **Retain Card 29 when it is the only offer, but suppress or deprioritise it when at least one alternative card is available.**

The counterfactual will investigate:

1. which Card 29 clickers are actually affected by this intervention;
2. what their choice set looks like after Card 29 is removed;
3. how customers historically behave in comparable panels without Card 29;
4. how much customer transfer is required for the intervention to break even.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

DB_PATH = Path("../data/processed/ocean_finance.duckdb")

con = duckdb.connect(str(DB_PATH), read_only=True)

quotes = con.execute("""
    SELECT *
    FROM quotes
""").df()

sales = con.execute("""
    SELECT *
    FROM sales
""").df()

# Convert values like "Card 29" -> 29
quotes["card_id"] = (
    quotes["card_id"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

sales["card_id"] = (
    sales["card_id"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

print("Quotes:", quotes.shape)
print("Sales:", sales.shape)
print()

print(quotes.dtypes)
print()

print("Example card IDs:")
print(quotes["card_id"].unique()[:10])

Quotes: (9945, 7)
Sales: (34, 3)

application_id         str
card_id              int64
clicked              int64
apr                float64
apr_rank             int64
likelihood           int64
likelihood_rank      int64
dtype: object

Example card IDs:
[ 1  2  3  4  7  8  9 10 11 12]


In [2]:
card29_clickers = (
    quotes.loc[
        (quotes["card_id"] == 29) &
        (quotes["clicked"] == 1),
        "application_id"
    ]
    .drop_duplicates()
)

print("Unique Card 29 clickers:", len(card29_clickers))


Unique Card 29 clickers: 398


In [3]:
print("Data types:")
print(quotes.dtypes)
print()

print("Unique clicked values:")
print(quotes["clicked"].unique())
print()

print("Card 29 rows:")
print(
    quotes.loc[quotes["card_id"] == 29, "clicked"]
    .value_counts(dropna=False)
)
print()

print(
    "Card 29 total offer rows:",
    (quotes["card_id"] == 29).sum()
)

print(
    "Card 29 clicked rows:",
    (
        (quotes["card_id"] == 29) &
        (quotes["clicked"] == 1)
    ).sum()
)

print(
    "Unique applications that clicked Card 29:",
    quotes.loc[
        (quotes["card_id"] == 29) &
        (quotes["clicked"] == 1),
        "application_id"
    ].nunique()
)

Data types:
application_id         str
card_id              int64
clicked              int64
apr                float64
apr_rank             int64
likelihood           int64
likelihood_rank      int64
dtype: object

Unique clicked values:
[0 1]

Card 29 rows:
clicked
0    1198
1     398
Name: count, dtype: int64

Card 29 total offer rows: 1596
Card 29 clicked rows: 398
Unique applications that clicked Card 29: 398


In [4]:
card29_panels = (
    quotes[
        quotes["application_id"].isin(card29_clickers)
    ]
    .copy()
)

print("Applications:", card29_panels["application_id"].nunique())
print("Offer rows:", len(card29_panels))

Applications: 398
Offer rows: 1116


In [5]:
def get_alternatives(group):
    alternatives = (
        group.loc[group["card_id"] != 29, "card_id"]
        .astype(int)
        .sort_values()
        .tolist()
    )

    return tuple(alternatives)


alternative_sets = (
    card29_panels
    .groupby("application_id")
    .apply(get_alternatives, include_groups=False)
    .rename("alternative_cards")
    .reset_index()
)

alternative_sets["alternative_count"] = (
    alternative_sets["alternative_cards"].apply(len)
)

alternative_sets.head(10)

,application_id,alternative_cards,alternative_count
0,K1lxV3FRb1hzRkF2ZXBVZEFqR1U2Zz090,"(14, 15)",2
1,K3BjYVRrSzJ0ZmZyamJscWJHU2ZPdz090,"(15, 16)",2
2,K3BlSGNadjg4bnl1SUZnWElXdGp0QT090,"(30, 31)",2
3,K3E4cXh1QVNYRThaYjJaUHVoSjZydz090,"(31,)",1
4,K3ErTUZMZWMxMEpaMGE3YTRVSEJlUT090,"(30, 31)",2
5,KzBGdVI1YVkxK3pMSjlNcVFsbE1Xdz090,"(30, 31)",2
6,KzF3ekZBMzF4dC9BaFNKR3ArNmxvQT090,"(30, 31)",2
7,L05nTVhkRFR4WXIwS285ckNJK2ZHUT090,(),0
8,L0RlVG5nbG5IdTFCU0VNcEVaekRCQT090,"(30, 31)",2
9,L0pqbzN3bTJPUFBxanRpSzBxY24yUT090,"(30, 31)",2


In [6]:
alternative_sets["intervention_group"] = np.where(
    alternative_sets["alternative_count"] == 0,
    "retain_card29",
    "suppress_card29"
)

intervention_summary = (
    alternative_sets["intervention_group"]
    .value_counts()
    .rename_axis("group")
    .reset_index(name="applications")
)

intervention_summary["share"] = (
    intervention_summary["applications"] /
    len(alternative_sets)
)

intervention_summary

,group,applications,share
0,suppress_card29,315,0.791457
1,retain_card29,83,0.208543


In [7]:
n_total = len(alternative_sets)

n_retain = (
    alternative_sets["alternative_count"] == 0
).sum()

n_suppress = (
    alternative_sets["alternative_count"] > 0
).sum()

print(f"Total Card 29 clickers: {n_total}")
print(f"Retain Card 29:        {n_retain}")
print(f"Suppress Card 29:      {n_suppress}")
print()
print(
    f"Suppression population: "
    f"{n_suppress / n_total:.1%}"
)

assert n_total == 398
assert n_retain == 83
assert n_suppress == 315
assert n_retain + n_suppress == n_total

Total Card 29 clickers: 398
Retain Card 29:        83
Suppress Card 29:      315

Suppression population: 79.1%


In [8]:
alternative_distribution = (
    alternative_sets.loc[
        alternative_sets["intervention_group"] == "suppress_card29",
        "alternative_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("alternative_count")
    .reset_index(name="applications")
)

alternative_distribution["share"] = (
    alternative_distribution["applications"] /
    alternative_distribution["applications"].sum()
)

alternative_distribution

,alternative_count,applications,share
0,1,42,0.133333
1,2,198,0.628571
2,3,41,0.130159
3,4,19,0.060317
4,5,11,0.034921
5,6,2,0.006349
6,7,2,0.006349


In [9]:
rank_check = quotes.copy()

# Test competition/min ranking
rank_check["likelihood_rank_min"] = (
    rank_check
    .groupby("application_id")["likelihood"]
    .rank(method="min", ascending=False)
)

rank_check["apr_rank_min"] = (
    rank_check
    .groupby("application_id")["apr"]
    .rank(method="min", ascending=True)
)

# Test dense ranking
rank_check["likelihood_rank_dense"] = (
    rank_check
    .groupby("application_id")["likelihood"]
    .rank(method="dense", ascending=False)
)

rank_check["apr_rank_dense"] = (
    rank_check
    .groupby("application_id")["apr"]
    .rank(method="dense", ascending=True)
)

print("Likelihood ranking")
print(
    "Min match:",
    (rank_check["likelihood_rank"] == rank_check["likelihood_rank_min"]).mean()
)
print(
    "Dense match:",
    (rank_check["likelihood_rank"] == rank_check["likelihood_rank_dense"]).mean()
)

print()

print("APR ranking")
print(
    "Min match:",
    (rank_check["apr_rank"] == rank_check["apr_rank_min"]).mean()
)
print(
    "Dense match:",
    (rank_check["apr_rank"] == rank_check["apr_rank_dense"]).mean()
)

Likelihood ranking
Min match: 1.0
Dense match: 0.7660130718954249

APR ranking
Min match: 1.0
Dense match: 0.7751633986928105


In [10]:
affected_ids = alternative_sets.loc[
    alternative_sets["intervention_group"] == "suppress_card29",
    "application_id"
]

post_removal = (
    card29_panels[
        card29_panels["application_id"].isin(affected_ids) &
        (card29_panels["card_id"] != 29)
    ]
    .copy()
)

print("Affected applications:", post_removal["application_id"].nunique())
print("Remaining offer rows:", len(post_removal))

assert post_removal["application_id"].nunique() == 315
assert 29 not in post_removal["card_id"].unique()

Affected applications: 315
Remaining offer rows: 718


In [11]:
post_removal["likelihood_rank_after"] = (
    post_removal
    .groupby("application_id")["likelihood"]
    .rank(method="min", ascending=False)
    .astype(int)
)

post_removal["apr_rank_after"] = (
    post_removal
    .groupby("application_id")["apr"]
    .rank(method="min", ascending=True)
    .astype(int)
)

post_removal[
    [
        "application_id",
        "card_id",
        "likelihood",
        "likelihood_rank_after",
        "apr",
        "apr_rank_after"
    ]
].head(15)

,application_id,card_id,likelihood,likelihood_rank_after,apr,apr_rank_after
327,bzNjS3d5ZG5uVFd1NWI2THd5dGkvdz090,30,90,1,32.9,1
328,bzNjS3d5ZG5uVFd1NWI2THd5dGkvdz090,31,90,1,32.9,1
340,aHlFbkFDVjFRVjlyMU0wWWZQeTA0dz090,15,100,1,39.9,1
341,aHlFbkFDVjFRVjlyMU0wWWZQeTA0dz090,14,100,1,59.9,2
419,NmYzSnYvbGxqV1B2a3IrM2RxdEIwZz090,25,100,1,33.9,3
421,NmYzSnYvbGxqV1B2a3IrM2RxdEIwZz090,31,95,2,32.9,1
422,NmYzSnYvbGxqV1B2a3IrM2RxdEIwZz090,30,95,2,32.9,1
502,b09BbGE1TCtoL0l2VVRwLzNuTzFLUT090,15,100,1,39.9,1
503,b09BbGE1TCtoL0l2VVRwLzNuTzFLUT090,14,100,1,59.9,2
532,ckFmYnpyUExranBYSFNLU1JuU0JGQT090,24,100,1,39.9,1


In [12]:
rank1_summary = (
    post_removal[
        post_removal["likelihood_rank_after"] == 1
    ]
    .groupby("application_id")
    .agg(
        n_rank1_cards=("card_id", "nunique")
    )
    .reset_index()
)

rank1_distribution = (
    rank1_summary["n_rank1_cards"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_rank1_cards")
    .reset_index(name="applications")
)

rank1_distribution["share"] = (
    rank1_distribution["applications"] /
    rank1_distribution["applications"].sum()
)

rank1_distribution

,number_of_rank1_cards,applications,share
0,1,122,0.387302
1,2,183,0.580952
2,3,10,0.031746


In [13]:
new_rank1_cards = (
    post_removal[
        post_removal["likelihood_rank_after"] == 1
    ]
    .groupby("card_id")
    .agg(
        rank1_appearances=("application_id", "nunique")
    )
    .sort_values("rank1_appearances", ascending=False)
    .reset_index()
)

new_rank1_cards["share_of_315"] = (
    new_rank1_cards["rank1_appearances"] / 315
)

new_rank1_cards.head(15)

,card_id,rank1_appearances,share_of_315
0,30,203,0.644444
1,31,171,0.542857
2,14,25,0.079365
3,7,17,0.053968
4,15,15,0.047619
5,23,14,0.044444
6,9,13,0.041270
7,25,10,0.031746
8,10,9,0.028571
9,8,8,0.025397


In [14]:
rank1_sets = (
    post_removal[
        post_removal["likelihood_rank_after"] == 1
    ]
    .groupby("application_id")["card_id"]
    .agg(lambda x: tuple(sorted(x.astype(int).unique())))
    .rename("rank1_cards")
    .reset_index()
)

rank1_set_distribution = (
    rank1_sets["rank1_cards"]
    .value_counts()
    .rename_axis("rank1_cards")
    .reset_index(name="applications")
)

rank1_set_distribution["share_of_315"] = (
    rank1_set_distribution["applications"] / 315
)

rank1_set_distribution.head(15)

,rank1_cards,applications,share_of_315
0,"(30, 31)",156,0.495238
1,"(30,)",47,0.149206
2,"(31,)",15,0.047619
3,"(7,)",11,0.034921
4,"(23,)",10,0.031746
5,"(14,)",10,0.031746
6,"(24,)",7,0.022222
7,"(9,)",6,0.019048
8,"(14, 15)",5,0.015873
9,"(15, 16)",4,0.012698


In [15]:
# Applications that contained Card 29 anywhere
apps_with_29 = set(
    quotes.loc[
        quotes["card_id"] == 29,
        "application_id"
    ]
)

# Historical applications where Card 29 was completely absent
no29 = quotes[
    ~quotes["application_id"].isin(apps_with_29)
].copy()

print("Applications without Card 29:", no29["application_id"].nunique())
print("Offer rows:", len(no29))

assert 29 not in no29["card_id"].unique()

Applications without Card 29: 2088
Offer rows: 3529


In [16]:
no29_app_behaviour = (
    no29
    .groupby("application_id")
    .agg(
        offers=("card_id", "nunique"),
        clicks=("clicked", "sum")
    )
    .reset_index()
)

no29_app_behaviour["any_click"] = (
    no29_app_behaviour["clicks"] > 0
)

print("Applications:", len(no29_app_behaviour))
print(
    "Any click:",
    no29_app_behaviour["any_click"].mean()
)
print(
    "No click:",
    1 - no29_app_behaviour["any_click"].mean()
)

print()
print("Number of clicks per application:")
print(
    no29_app_behaviour["clicks"]
    .value_counts()
    .sort_index()
)

Applications: 2088
Any click: 0.5081417624521073
No click: 0.4918582375478927

Number of clicks per application:
clicks
0    1027
1    1050
2      10
3       1
Name: count, dtype: int64


### A. Offer level CTR by rank

In [17]:
rank_click_rates = (
    no29
    .groupby("likelihood_rank")
    .agg(
        offers=("card_id", "size"),
        clicks=("clicked", "sum")
    )
    .reset_index()
)

rank_click_rates["offer_ctr"] = (
    rank_click_rates["clicks"] /
    rank_click_rates["offers"]
)

rank_click_rates.head(10)

,likelihood_rank,offers,clicks,offer_ctr
0,1,2982,1026,0.344064
1,2,236,34,0.144068
2,3,124,9,0.072581
3,4,77,3,0.038961
4,5,32,1,0.031250
5,6,31,0,0.000000
6,7,12,0,0.000000
7,8,15,0,0.000000
8,9,11,0,0.000000
9,10,4,0,0.000000


### B. Share of actual clicks going by rank

In [18]:
clicked_no29 = no29[
    no29["clicked"] == 1
].copy()

click_share_by_rank = (
    clicked_no29
    .groupby("likelihood_rank")
    .size()
    .rename("clicks")
    .reset_index()
)

click_share_by_rank["share_of_clicks"] = (
    click_share_by_rank["clicks"] /
    click_share_by_rank["clicks"].sum()
)

click_share_by_rank.head(10)

,likelihood_rank,clicks,share_of_clicks
0,1,1026,0.956198
1,2,34,0.031687
2,3,9,0.008388
3,4,3,0.002796
4,5,1,0.000932


In [19]:
no29_rank1_counts = (
    no29[
        no29["likelihood_rank"] == 1
    ]
    .groupby("application_id")
    .agg(
        n_rank1_cards=("card_id", "nunique")
    )
    .reset_index()
)

no29_rank1_behaviour = (
    no29_app_behaviour
    .merge(
        no29_rank1_counts,
        on="application_id",
        how="left"
    )
)

rank1_count_clickout = (
    no29_rank1_behaviour
    .groupby("n_rank1_cards")
    .agg(
        applications=("application_id", "nunique"),
        applications_with_click=("any_click", "sum")
    )
    .reset_index()
)

rank1_count_clickout["clickout_rate"] = (
    rank1_count_clickout["applications_with_click"] /
    rank1_count_clickout["applications"]
)

rank1_count_clickout

,n_rank1_cards,applications,applications_with_click,clickout_rate
0,1,1397,718,0.513958
1,2,519,272,0.524085
2,3,147,62,0.421769
3,4,19,7,0.368421
4,5,6,2,0.333333


In [20]:
# Applications without Card 29 where Cards 30 and 31
# are both present and both rank 1

apps_30_rank1 = set(
    no29.loc[
        (no29["card_id"] == 30) &
        (no29["likelihood_rank"] == 1),
        "application_id"
    ]
)

apps_31_rank1 = set(
    no29.loc[
        (no29["card_id"] == 31) &
        (no29["likelihood_rank"] == 1),
        "application_id"
    ]
)

apps_30_31_tied = apps_30_rank1 & apps_31_rank1

historical_30_31 = no29[
    no29["application_id"].isin(apps_30_31_tied)
].copy()

print(
    "Historical applications with 30 & 31 both rank 1:",
    historical_30_31["application_id"].nunique()
)

Historical applications with 30 & 31 both rank 1: 127


In [21]:
apps_30_31 = sorted(apps_30_31_tied)

behaviour_30_31 = pd.DataFrame({
    "application_id": apps_30_31
})

clicked_apps = set(
    historical_30_31.loc[
        historical_30_31["clicked"] == 1,
        "application_id"
    ]
)

clicked_30_apps = set(
    historical_30_31.loc[
        (historical_30_31["card_id"] == 30) &
        (historical_30_31["clicked"] == 1),
        "application_id"
    ]
)

clicked_31_apps = set(
    historical_30_31.loc[
        (historical_30_31["card_id"] == 31) &
        (historical_30_31["clicked"] == 1),
        "application_id"
    ]
)

clicked_other_apps = set(
    historical_30_31.loc[
        (~historical_30_31["card_id"].isin([30, 31])) &
        (historical_30_31["clicked"] == 1),
        "application_id"
    ]
)

behaviour_30_31["any_click"] = (
    behaviour_30_31["application_id"].isin(clicked_apps)
)

behaviour_30_31["clicked_30"] = (
    behaviour_30_31["application_id"].isin(clicked_30_apps)
)

behaviour_30_31["clicked_31"] = (
    behaviour_30_31["application_id"].isin(clicked_31_apps)
)

behaviour_30_31["clicked_other"] = (
    behaviour_30_31["application_id"].isin(clicked_other_apps)
)

print("Applications:", len(behaviour_30_31))
print(f"Any click:     {behaviour_30_31['any_click'].mean():.1%}")
print(f"No click:      {(~behaviour_30_31['any_click']).mean():.1%}")
print(f"Clicked 30:    {behaviour_30_31['clicked_30'].mean():.1%}")
print(f"Clicked 31:    {behaviour_30_31['clicked_31'].mean():.1%}")
print(f"Clicked other: {behaviour_30_31['clicked_other'].mean():.1%}")

Applications: 127
Any click:     59.8%
No click:      40.2%
Clicked 30:    47.2%
Clicked 31:    11.8%
Clicked other: 0.8%


In [22]:
no29_rank1_sets = (
    no29[
        no29["likelihood_rank"] == 1
    ]
    .groupby("application_id")["card_id"]
    .agg(lambda x: tuple(sorted(x.astype(int).unique())))
    .rename("rank1_cards")
    .reset_index()
)

exact_30_31_apps = set(
    no29_rank1_sets.loc[
        no29_rank1_sets["rank1_cards"] == (30, 31),
        "application_id"
    ]
)

exact_30_31 = no29[
    no29["application_id"].isin(exact_30_31_apps)
].copy()

print(
    "Historical applications with exact rank-1 set (30, 31):",
    len(exact_30_31_apps)
)

Historical applications with exact rank-1 set (30, 31): 126


In [23]:
exact_apps = sorted(exact_30_31_apps)

exact_behaviour = pd.DataFrame({
    "application_id": exact_apps
})

exact_clicked = set(
    exact_30_31.loc[
        exact_30_31["clicked"] == 1,
        "application_id"
    ]
)

exact_clicked_30 = set(
    exact_30_31.loc[
        (exact_30_31["card_id"] == 30) &
        (exact_30_31["clicked"] == 1),
        "application_id"
    ]
)

exact_clicked_31 = set(
    exact_30_31.loc[
        (exact_30_31["card_id"] == 31) &
        (exact_30_31["clicked"] == 1),
        "application_id"
    ]
)

exact_clicked_other = set(
    exact_30_31.loc[
        (~exact_30_31["card_id"].isin([30, 31])) &
        (exact_30_31["clicked"] == 1),
        "application_id"
    ]
)

exact_behaviour["any_click"] = (
    exact_behaviour["application_id"].isin(exact_clicked)
)

exact_behaviour["clicked_30"] = (
    exact_behaviour["application_id"].isin(exact_clicked_30)
)

exact_behaviour["clicked_31"] = (
    exact_behaviour["application_id"].isin(exact_clicked_31)
)

exact_behaviour["clicked_other"] = (
    exact_behaviour["application_id"].isin(exact_clicked_other)
)

print("Applications:", len(exact_behaviour))
print(f"Any click:     {exact_behaviour['any_click'].mean():.1%}")
print(f"No click:      {(~exact_behaviour['any_click']).mean():.1%}")
print(f"Clicked 30:    {exact_behaviour['clicked_30'].mean():.1%}")
print(f"Clicked 31:    {exact_behaviour['clicked_31'].mean():.1%}")
print(f"Clicked other: {exact_behaviour['clicked_other'].mean():.1%}")

Applications: 126
Any click:     59.5%
No click:      40.5%
Clicked 30:    47.6%
Clicked 31:    11.9%
Clicked other: 0.0%


In [24]:
post_removal_choice_sets = (
    post_removal
    .groupby("application_id")["card_id"]
    .agg(lambda x: tuple(sorted(x.astype(int).unique())))
    .rename("remaining_cards")
    .reset_index()
)

post_removal_choice_sets.head()

,application_id,remaining_cards
0,K1lxV3FRb1hzRkF2ZXBVZEFqR1U2Zz090,"(14, 15)"
1,K3BjYVRrSzJ0ZmZyamJscWJHU2ZPdz090,"(15, 16)"
2,K3BlSGNadjg4bnl1SUZnWElXdGp0QT090,"(30, 31)"
3,K3E4cXh1QVNYRThaYjJaUHVoSjZydz090,"(31,)"
4,K3ErTUZMZWMxMEpaMGE3YTRVSEJlUT090,"(30, 31)"


In [25]:
historical_no29_choice_sets = (
    no29
    .groupby("application_id")["card_id"]
    .agg(lambda x: tuple(sorted(x.astype(int).unique())))
    .rename("remaining_cards")
    .reset_index()
)

historical_choice_set_counts = (
    historical_no29_choice_sets["remaining_cards"]
    .value_counts()
    .rename_axis("remaining_cards")
    .reset_index(name="historical_applications")
)

historical_choice_set_counts.head(10)

,remaining_cards,historical_applications
0,"(7,)",177
1,"(9,)",142
2,"(23,)",138
3,"(30,)",130
4,"(30, 31)",127
5,"(24,)",120
6,"(4,)",115
7,"(14,)",112
8,"(10,)",103
9,"(25,)",80


In [26]:
choice_set_match = (
    post_removal_choice_sets
    .merge(
        historical_choice_set_counts,
        on="remaining_cards",
        how="left"
    )
)

choice_set_match["historical_applications"] = (
    choice_set_match["historical_applications"]
    .fillna(0)
    .astype(int)
)

choice_set_match["exact_match_available"] = (
    choice_set_match["historical_applications"] > 0
)

print(
    "Affected applications:",
    len(choice_set_match)
)

print(
    "Exact historical match:",
    choice_set_match["exact_match_available"].sum()
)

print(
    "No exact match:",
    (~choice_set_match["exact_match_available"]).sum()
)

print(
    "Exact-match coverage:",
    f"{choice_set_match['exact_match_available'].mean():.1%}"
)

Affected applications: 315
Exact historical match: 279
No exact match: 36
Exact-match coverage: 88.6%


In [27]:
match_strength = pd.cut(
    choice_set_match["historical_applications"],
    bins=[-1, 0, 1, 4, 9, float("inf")],
    labels=[
        "No match",
        "1 historical app",
        "2–4 historical apps",
        "5–9 historical apps",
        "10+ historical apps"
    ]
)

match_strength.value_counts().reindex([
    "No match",
    "1 historical app",
    "2–4 historical apps",
    "5–9 historical apps",
    "10+ historical apps"
])

historical_applications
No match                36
1 historical app        12
2–4 historical apps     35
5–9 historical apps     15
10+ historical apps    217
Name: count, dtype: int64

In [28]:
affected_state = (
    post_removal_choice_sets
    .merge(
        rank1_sets,
        on="application_id",
        how="left"
    )
)

affected_state.head()

,application_id,remaining_cards,rank1_cards
0,K1lxV3FRb1hzRkF2ZXBVZEFqR1U2Zz090,"(14, 15)","(14, 15)"
1,K3BjYVRrSzJ0ZmZyamJscWJHU2ZPdz090,"(15, 16)","(15, 16)"
2,K3BlSGNadjg4bnl1SUZnWElXdGp0QT090,"(30, 31)","(30,)"
3,K3E4cXh1QVNYRThaYjJaUHVoSjZydz090,"(31,)","(31,)"
4,K3ErTUZMZWMxMEpaMGE3YTRVSEJlUT090,"(30, 31)","(30,)"


In [29]:
historical_state = (
    historical_no29_choice_sets
    .merge(
        no29_rank1_sets,
        on="application_id",
        how="left"
    )
)

historical_state.head()

,application_id,remaining_cards,rank1_cards
0,K053bWNzZCtIQkVpVVFCZlJtZHhQZz090,"(25,)","(25,)"
1,K05nbTJtS1RGZ1VId0UyTERoMlJSZz090,"(2, 3, 10)","(2, 3, 10)"
2,K091c2xITXQ1VE1lMXYzMEx2c3gwZz090,"(4,)","(4,)"
3,K09WWVZiQXFnaHN6eVJ3OUtTMFBBQT090,"(15, 16)","(15, 16)"
4,K0hJdTlzWnRHQzFzMGR6ZjhWV3ZNUT090,"(7,)","(7,)"


In [30]:
historical_state_counts = (
    historical_state
    .groupby(
        ["remaining_cards", "rank1_cards"]
    )
    .size()
    .rename("historical_applications")
    .reset_index()
)

state_match = (
    affected_state
    .merge(
        historical_state_counts,
        on=["remaining_cards", "rank1_cards"],
        how="left"
    )
)

state_match["historical_applications"] = (
    state_match["historical_applications"]
    .fillna(0)
    .astype(int)
)

state_match["state_match_available"] = (
    state_match["historical_applications"] > 0
)

print("Affected applications:", len(state_match))
print(
    "Exact choice + rank-state match:",
    state_match["state_match_available"].sum()
)
print(
    "No exact state match:",
    (~state_match["state_match_available"]).sum()
)
print(
    "Coverage:",
    f"{state_match['state_match_available'].mean():.1%}"
)

Affected applications: 315
Exact choice + rank-state match: 278
No exact state match: 37
Coverage: 88.3%


In [31]:
state_match_strength = pd.cut(
    state_match["historical_applications"],
    bins=[-1, 0, 1, 4, 9, float("inf")],
    labels=[
        "No match",
        "1 historical app",
        "2–4 historical apps",
        "5–9 historical apps",
        "10+ historical apps"
    ]
)

state_match_strength.value_counts().reindex([
    "No match",
    "1 historical app",
    "2–4 historical apps",
    "5–9 historical apps",
    "10+ historical apps"
])

historical_applications
No match                37
1 historical app        44
2–4 historical apps     35
5–9 historical apps     15
10+ historical apps    184
Name: count, dtype: int64

In [32]:
historical_rank1_set_counts = (
    no29_rank1_sets["rank1_cards"]
    .value_counts()
    .rename_axis("rank1_cards")
    .reset_index(name="historical_applications")
)

rank1_set_match = (
    rank1_sets
    .merge(
        historical_rank1_set_counts,
        on="rank1_cards",
        how="left"
    )
)

rank1_set_match["historical_applications"] = (
    rank1_set_match["historical_applications"]
    .fillna(0)
    .astype(int)
)

rank1_set_match["match_available"] = (
    rank1_set_match["historical_applications"] > 0
)

print("Affected applications:", len(rank1_set_match))
print(
    "Rank-1 set historical match:",
    rank1_set_match["match_available"].sum()
)
print(
    "No rank-1 set match:",
    (~rank1_set_match["match_available"]).sum()
)
print(
    "Coverage:",
    f"{rank1_set_match['match_available'].mean():.1%}"
)

Affected applications: 315
Rank-1 set historical match: 312
No rank-1 set match: 3
Coverage: 99.0%


In [33]:
rank1_set_strength = pd.cut(
    rank1_set_match["historical_applications"],
    bins=[-1, 0, 1, 4, 9, float("inf")],
    labels=[
        "No match",
        "1 historical app",
        "2–4 historical apps",
        "5–9 historical apps",
        "10+ historical apps"
    ]
)

rank1_set_strength.value_counts().reindex([
    "No match",
    "1 historical app",
    "2–4 historical apps",
    "5–9 historical apps",
    "10+ historical apps"
])

historical_applications
No match                 3
1 historical app         2
2–4 historical apps     23
5–9 historical apps      5
10+ historical apps    282
Name: count, dtype: int64

In [34]:
rank1_set_reference = (
    rank1_sets["rank1_cards"]
    .value_counts()
    .rename_axis("rank1_cards")
    .reset_index(name="affected_applications")
    .merge(
        historical_rank1_set_counts,
        on="rank1_cards",
        how="left"
    )
)

rank1_set_reference["historical_applications"] = (
    rank1_set_reference["historical_applications"]
    .fillna(0)
    .astype(int)
)

rank1_set_reference["affected_share"] = (
    rank1_set_reference["affected_applications"] / 315
)

rank1_set_reference.head(15)

,rank1_cards,affected_applications,historical_applications,affected_share
0,"(30, 31)",156,126,0.495238
1,"(30,)",47,133,0.149206
2,"(31,)",15,4,0.047619
3,"(7,)",11,200,0.034921
4,"(23,)",10,155,0.031746
5,"(14,)",10,124,0.031746
6,"(24,)",7,126,0.022222
7,"(9,)",6,158,0.019048
8,"(14, 15)",5,48,0.015873
9,"(15, 16)",4,44,0.012698


In [35]:
no29_behaviour = (
    no29
    .groupby("application_id")
    .agg(
        total_clicks=("clicked", "sum")
    )
    .reset_index()
)

no29_behaviour["any_click"] = (
    no29_behaviour["total_clicks"] > 0
)

In [36]:
no29_rank1_behaviour = (
    no29_rank1_sets
    .merge(
        no29_behaviour,
        on="application_id",
        how="left"
    )
)

no29_rank1_behaviour.head()

,application_id,rank1_cards,total_clicks,any_click
0,K053bWNzZCtIQkVpVVFCZlJtZHhQZz090,"(25,)",0,False
1,K05nbTJtS1RGZ1VId0UyTERoMlJSZz090,"(2, 3, 10)",1,True
2,K091c2xITXQ1VE1lMXYzMEx2c3gwZz090,"(4,)",0,False
3,K09WWVZiQXFnaHN6eVJ3OUtTMFBBQT090,"(15, 16)",1,True
4,K0hJdTlzWnRHQzFzMGR6ZjhWV3ZNUT090,"(7,)",1,True


In [37]:
rank1_behaviour_summary = (
    no29_rank1_behaviour
    .groupby("rank1_cards")
    .agg(
        historical_applications=("application_id", "nunique"),
        applications_with_click=("any_click", "sum"),
        total_clicks=("total_clicks", "sum")
    )
    .reset_index()
)

rank1_behaviour_summary["clickout_rate"] = (
    rank1_behaviour_summary["applications_with_click"] /
    rank1_behaviour_summary["historical_applications"]
)

rank1_behaviour_summary["clicks_per_application"] = (
    rank1_behaviour_summary["total_clicks"] /
    rank1_behaviour_summary["historical_applications"]
)

In [38]:
affected_rank1_behaviour = (
    rank1_set_reference
    .merge(
        rank1_behaviour_summary[
            [
                "rank1_cards",
                "clickout_rate",
                "clicks_per_application"
            ]
        ],
        on="rank1_cards",
        how="left"
    )
)

affected_rank1_behaviour.head(15)

,rank1_cards,affected_applications,historical_applications,affected_share,clickout_rate,clicks_per_application
0,"(30, 31)",156,126,0.495238,0.595238,0.595238
1,"(30,)",47,133,0.149206,0.661654,0.661654
2,"(31,)",15,4,0.047619,0.750000,0.750000
3,"(7,)",11,200,0.034921,0.575000,0.585000
4,"(23,)",10,155,0.031746,0.522581,0.522581
5,"(14,)",10,124,0.031746,0.411290,0.419355
6,"(24,)",7,126,0.022222,0.547619,0.547619
7,"(9,)",6,158,0.019048,0.493671,0.493671
8,"(14, 15)",5,48,0.015873,0.541667,0.562500
9,"(15, 16)",4,44,0.012698,0.431818,0.431818


In [39]:
supported_rank1 = affected_rank1_behaviour[
    affected_rank1_behaviour["historical_applications"] >= 10
].copy()

unsupported_rank1 = affected_rank1_behaviour[
    affected_rank1_behaviour["historical_applications"] < 10
].copy()

print(
    "Empirically supported affected applications:",
    supported_rank1["affected_applications"].sum()
)

print(
    "Residual affected applications:",
    unsupported_rank1["affected_applications"].sum()
)

print()

supported_rank1[
    [
        "rank1_cards",
        "affected_applications",
        "historical_applications",
        "clickout_rate",
        "clicks_per_application"
    ]
].head(20)

Empirically supported affected applications: 282
Residual affected applications: 33



,rank1_cards,affected_applications,historical_applications,clickout_rate,clicks_per_application
0,"(30, 31)",156,126,0.595238,0.595238
1,"(30,)",47,133,0.661654,0.661654
3,"(7,)",11,200,0.575000,0.585000
4,"(23,)",10,155,0.522581,0.522581
5,"(14,)",10,124,0.411290,0.419355
6,"(24,)",7,126,0.547619,0.547619
7,"(9,)",6,158,0.493671,0.493671
8,"(14, 15)",5,48,0.541667,0.562500
9,"(15, 16)",4,44,0.431818,0.431818
10,"(25,)",4,99,0.353535,0.353535


In [40]:
supported_sets = set(
    supported_rank1["rank1_cards"]
)

choice_rows = []

for rank1_set in supported_sets:

    # Historical applications with this exact rank-1 configuration
    apps = set(
        no29_rank1_sets.loc[
            no29_rank1_sets["rank1_cards"] == rank1_set,
            "application_id"
        ]
    )

    hist = no29[
        no29["application_id"].isin(apps)
    ].copy()

    n_apps = len(apps)

    # Any click
    apps_with_click = set(
        hist.loc[
            hist["clicked"] == 1,
            "application_id"
        ]
    )

    # Clicks on rank-1 cards
    rank1_clicks = hist[
        (hist["clicked"] == 1) &
        (hist["card_id"].isin(rank1_set))
    ]

    # Clicks on lower-ranked cards
    lower_rank_clicks = hist[
        (hist["clicked"] == 1) &
        (~hist["card_id"].isin(rank1_set))
    ]

    row = {
        "rank1_cards": rank1_set,
        "historical_apps": n_apps,
        "any_click_rate": len(apps_with_click) / n_apps,
        "no_click_rate": 1 - (len(apps_with_click) / n_apps),
        "lower_rank_clicks_per_app": len(lower_rank_clicks) / n_apps,
    }

    # Probability each top-ranked card is clicked
    for card in rank1_set:
        card_clicks = (
            rank1_clicks["card_id"] == card
        ).sum()

        row[f"card_{card}_clicks_per_app"] = (
            card_clicks / n_apps
        )

    choice_rows.append(row)


rank1_choice_behaviour = (
    pd.DataFrame(choice_rows)
    .sort_values(
        "historical_apps",
        ascending=False
    )
    .reset_index(drop=True)
)

rank1_choice_behaviour.head(20)

,rank1_cards,historical_apps,any_click_rate,no_click_rate,lower_rank_clicks_per_app,card_8_clicks_per_app,card_14_clicks_per_app,card_23_clicks_per_app,card_9_clicks_per_app,card_4_clicks_per_app,...,card_10_clicks_per_app,card_25_clicks_per_app,card_15_clicks_per_app,card_22_clicks_per_app,card_16_clicks_per_app,card_24_clicks_per_app,card_30_clicks_per_app,card_2_clicks_per_app,card_3_clicks_per_app,card_31_clicks_per_app
0,"(7,)",200,0.575000,0.425000,0.030000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"(9,)",158,0.493671,0.506329,0.006329,NaN,NaN,NaN,0.487342,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"(23,)",155,0.522581,0.477419,0.058065,NaN,NaN,0.464516,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"(30,)",133,0.661654,0.338346,0.007519,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.654135,NaN,NaN,NaN
4,"(30, 31)",126,0.595238,0.404762,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.476190,NaN,NaN,0.119048
5,"(24,)",126,0.547619,0.452381,0.023810,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.52381,NaN,NaN,NaN,NaN
6,"(14,)",124,0.411290,0.588710,0.056452,NaN,0.362903,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,"(4,)",122,0.516393,0.483607,0.016393,NaN,NaN,NaN,NaN,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"(10,)",107,0.495327,0.504673,0.000000,NaN,NaN,NaN,NaN,NaN,...,0.495327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,"(25,)",99,0.353535,0.646465,0.060606,NaN,NaN,NaN,NaN,NaN,...,NaN,0.292929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
total_hist_apps = rank1_choice_behaviour["historical_apps"].sum()

weighted_lower_rank_clicks = (
    rank1_choice_behaviour["lower_rank_clicks_per_app"] *
    rank1_choice_behaviour["historical_apps"]
).sum()

print(
    "Lower-rank clicks across supported historical groups:",
    int(round(weighted_lower_rank_clicks))
)

print(
    "Lower-rank clicks per historical application:",
    weighted_lower_rank_clicks / total_hist_apps
)

Lower-rank clicks across supported historical groups: 39
Lower-rank clicks per historical application: 0.023172905525846704


In [42]:
supported_hist_ids = set()

for rank1_set in supported_sets:
    supported_hist_ids.update(
        no29_rank1_sets.loc[
            no29_rank1_sets["rank1_cards"] == rank1_set,
            "application_id"
        ]
    )

supported_hist = no29[
    no29["application_id"].isin(supported_hist_ids)
]

total_clicks_supported = (
    supported_hist["clicked"].sum()
)

lower_rank_clicks_supported = (
    supported_hist.loc[
        (supported_hist["clicked"] == 1) &
        (supported_hist["likelihood_rank"] != 1),
        "clicked"
    ].sum()
)

print("Total historical clicks:", total_clicks_supported)
print("Lower-rank clicks:", lower_rank_clicks_supported)

print(
    "Share of clicks below rank 1:",
    f"{lower_rank_clicks_supported / total_clicks_supported:.1%}"
)

Total historical clicks: 882
Lower-rank clicks: 39
Share of clicks below rank 1: 4.4%


In [43]:
important_sets = [
    (30, 31),
    (30,),
    (7,),
    (23,),
    (14,),
    (24,),
    (9,),
    (14, 15),
    (15, 16)
]

rank1_choice_behaviour[
    rank1_choice_behaviour["rank1_cards"].isin(important_sets)
]

,rank1_cards,historical_apps,any_click_rate,no_click_rate,lower_rank_clicks_per_app,card_8_clicks_per_app,card_14_clicks_per_app,card_23_clicks_per_app,card_9_clicks_per_app,card_4_clicks_per_app,...,card_10_clicks_per_app,card_25_clicks_per_app,card_15_clicks_per_app,card_22_clicks_per_app,card_16_clicks_per_app,card_24_clicks_per_app,card_30_clicks_per_app,card_2_clicks_per_app,card_3_clicks_per_app,card_31_clicks_per_app
0,"(7,)",200,0.575000,0.425000,0.030000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"(9,)",158,0.493671,0.506329,0.006329,NaN,NaN,NaN,0.487342,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"(23,)",155,0.522581,0.477419,0.058065,NaN,NaN,0.464516,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"(30,)",133,0.661654,0.338346,0.007519,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.654135,NaN,NaN,NaN
4,"(30, 31)",126,0.595238,0.404762,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.476190,NaN,NaN,0.119048
5,"(24,)",126,0.547619,0.452381,0.023810,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.52381,NaN,NaN,NaN,NaN
6,"(14,)",124,0.411290,0.588710,0.056452,NaN,0.362903,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,"(14, 15)",48,0.541667,0.458333,0.000000,NaN,0.104167,NaN,NaN,NaN,...,NaN,NaN,0.458333,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,"(15, 16)",44,0.431818,0.568182,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.272727,NaN,0.159091,NaN,NaN,NaN,NaN,NaN


In [44]:
supported_rank1_sets = set(
    supported_rank1["rank1_cards"]
)

prediction_rows = []

for _, affected_row in supported_rank1.iterrows():

    rank1_set = affected_row["rank1_cards"]
    n_affected = affected_row["affected_applications"]

    # Historical applications with the same rank-1 set
    hist_apps = set(
        no29_rank1_sets.loc[
            no29_rank1_sets["rank1_cards"] == rank1_set,
            "application_id"
        ]
    )

    hist = no29[
        no29["application_id"].isin(hist_apps)
    ].copy()

    n_hist = len(hist_apps)

    row = {
        "rank1_cards": rank1_set,
        "affected_applications": n_affected,
        "historical_applications": n_hist,
    }

    expected_top_rank_clicks = 0

    for card in rank1_set:

        historical_card_clicks = (
            (
                (hist["card_id"] == card) &
                (hist["clicked"] == 1)
            )
            .sum()
        )

        click_probability = historical_card_clicks / n_hist

        expected_clicks = (
            n_affected * click_probability
        )

        row[f"card_{card}_click_probability"] = click_probability
        row[f"card_{card}_expected_clicks"] = expected_clicks

        expected_top_rank_clicks += expected_clicks

    row["expected_top_rank_clicks"] = expected_top_rank_clicks

    row["expected_no_top_rank_click"] = (
        n_affected - expected_top_rank_clicks
    )

    prediction_rows.append(row)


supported_predictions = (
    pd.DataFrame(prediction_rows)
    .sort_values(
        "affected_applications",
        ascending=False
    )
    .reset_index(drop=True)
)

supported_predictions.head(20)

,rank1_cards,affected_applications,historical_applications,card_30_click_probability,card_30_expected_clicks,card_31_click_probability,card_31_expected_clicks,expected_top_rank_clicks,expected_no_top_rank_click,card_7_click_probability,...,card_8_click_probability,card_8_expected_clicks,card_22_click_probability,card_22_expected_clicks,card_10_click_probability,card_10_expected_clicks,card_2_click_probability,card_2_expected_clicks,card_3_click_probability,card_3_expected_clicks
0,"(30, 31)",156,126,0.476190,74.285714,0.119048,18.571429,92.857143,63.142857,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"(30,)",47,133,0.654135,30.744361,NaN,NaN,30.744361,16.255639,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"(7,)",11,200,NaN,NaN,NaN,NaN,6.105000,4.895000,0.555000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"(23,)",10,155,NaN,NaN,NaN,NaN,4.645161,5.354839,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"(14,)",10,124,NaN,NaN,NaN,NaN,3.629032,6.370968,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,"(24,)",7,126,NaN,NaN,NaN,NaN,3.666667,3.333333,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"(9,)",6,158,NaN,NaN,NaN,NaN,2.924051,3.075949,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,"(14, 15)",5,48,NaN,NaN,NaN,NaN,2.812500,2.187500,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"(4,)",4,122,NaN,NaN,NaN,NaN,2.000000,2.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,"(8, 9)",4,13,NaN,NaN,NaN,NaN,1.230769,2.769231,NaN,...,0.307692,1.230769,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
print(
    "Supported affected applications:",
    supported_predictions["affected_applications"].sum()
)

print(
    "Expected top-rank replacement clicks:",
    supported_predictions["expected_top_rank_clicks"].sum()
)

print(
    "Expected applications without a top-rank replacement click:",
    supported_predictions["expected_no_top_rank_click"].sum()
)

Supported affected applications: 282
Expected top-rank replacement clicks: 160.97198742330116
Expected applications without a top-rank replacement click: 121.02801257669886


In [46]:
card_expected_clicks = {}

for _, row in supported_predictions.iterrows():

    for card in row["rank1_cards"]:

        col = f"card_{card}_expected_clicks"

        card_expected_clicks[card] = (
            card_expected_clicks.get(card, 0)
            + row[col]
        )

expected_clicks_by_card = (
    pd.DataFrame(
        [
            {
                "card_id": card,
                "expected_clicks": clicks
            }
            for card, clicks in card_expected_clicks.items()
        ]
    )
    .sort_values(
        "expected_clicks",
        ascending=False
    )
    .reset_index(drop=True)
)

expected_clicks_by_card

,card_id,expected_clicks
0,30,105.030075
1,31,18.571429
2,7,7.465897
3,23,4.645161
4,14,4.216532
5,24,3.666667
6,15,3.382576
7,9,2.924051
8,8,2.421245
9,4,2.000000


In [47]:
clicks_by_card = (
    quotes
    .groupby("card_id")
    .agg(
        clicks=("clicked", "sum")
    )
    .reset_index()
)

card_perf = (
    clicks_by_card
    .merge(
        sales,
        on="card_id",
        how="left"
    )
)

card_perf["conversion_rate"] = (
    card_perf["sales"] /
    card_perf["clicks"]
)

card_perf["revenue_per_click"] = (
    card_perf["revenue"] /
    card_perf["clicks"]
)

card_perf[
    card_perf["card_id"].isin([29, 30, 31])
]

,card_id,clicks,sales,revenue,conversion_rate,revenue_per_click
28,29,398,63,2772.0,0.158291,6.964824
29,30,281,89,3916.0,0.316726,13.935943
30,31,80,21,924.0,0.262500,11.550000


In [48]:
supported_economics = (
    expected_clicks_by_card
    .merge(
        card_perf[
            [
                "card_id",
                "conversion_rate",
                "revenue_per_click"
            ]
        ],
        on="card_id",
        how="left"
    )
)

supported_economics["expected_sales"] = (
    supported_economics["expected_clicks"] *
    supported_economics["conversion_rate"]
)

supported_economics["expected_revenue"] = (
    supported_economics["expected_clicks"] *
    supported_economics["revenue_per_click"]
)

supported_economics = (
    supported_economics
    .sort_values(
        "expected_revenue",
        ascending=False
    )
    .reset_index(drop=True)
)

supported_economics

,card_id,expected_clicks,conversion_rate,revenue_per_click,expected_sales,expected_revenue
0,30,105.030075,0.316726,13.935943,33.265753,1463.693147
1,31,18.571429,0.262500,11.550000,4.875000,214.500000
2,7,7.465897,0.351351,11.945946,2.623153,89.187207
3,23,4.645161,0.326087,11.413043,1.514727,53.015428
4,8,2.421245,0.440000,19.800000,1.065348,47.940659
5,15,3.382576,0.469388,13.142857,1.587740,44.456710
6,24,3.666667,0.328947,11.513158,1.206140,42.214912
7,14,4.216532,0.241935,8.467742,1.020129,35.704507
8,9,2.924051,0.352459,11.983607,1.030608,35.040672
9,4,2.000000,0.357143,12.142857,0.714286,24.285714


In [49]:
supported_expected_clicks = (
    supported_economics["expected_clicks"].sum()
)

supported_expected_sales = (
    supported_economics["expected_sales"].sum()
)

supported_expected_revenue = (
    supported_economics["expected_revenue"].sum()
)

print(
    f"Expected replacement clicks: "
    f"{supported_expected_clicks:.2f}"
)

print(
    f"Expected replacement sales: "
    f"{supported_expected_sales:.2f}"
)

print(
    f"Expected replacement revenue: "
    f"£{supported_expected_revenue:,.2f}"
)

Expected replacement clicks: 160.97
Expected replacement sales: 51.45
Expected replacement revenue: £2,143.64


In [50]:
unsupported_rank1[
    [
        "rank1_cards",
        "affected_applications",
        "historical_applications",
        "clickout_rate",
        "clicks_per_application"
    ]
].sort_values(
    "affected_applications",
    ascending=False
)

,rank1_cards,affected_applications,historical_applications,clickout_rate,clicks_per_application
2,"(31,)",15,4,0.750000,0.750000
18,"(4, 14)",2,9,0.666667,0.666667
35,"(15, 16, 23)",1,4,0.250000,0.250000
34,"(14, 15, 25)",1,0,NaN,NaN
32,"(7, 10, 25)",1,1,0.000000,0.000000
31,"(2, 3, 10)",1,3,0.333333,0.333333
29,"(10, 15, 16)",1,2,0.500000,0.500000
28,"(14, 23)",1,4,0.500000,0.500000
27,"(9, 15, 16)",1,4,0.750000,0.750000
26,"(4, 14, 25)",1,2,0.500000,0.500000


In [51]:
clicks_by_card = (
    quotes
    .groupby("card_id")
    .agg(
        clicks=("clicked", "sum")
    )
    .reset_index()
)

card_perf = (
    clicks_by_card
    .merge(
        sales,
        on="card_id",
        how="left"
    )
)

card_perf["conversion_rate"] = (
    card_perf["sales"] /
    card_perf["clicks"]
)

card_perf["revenue_per_click"] = (
    card_perf["revenue"] /
    card_perf["clicks"]
)

print(
    card_perf[
        card_perf["card_id"].isin([29, 30, 31])
    ]
)

    card_id  clicks  sales  revenue  conversion_rate  revenue_per_click
28       29     398     63   2772.0         0.158291           6.964824
29       30     281     89   3916.0         0.316726          13.935943
30       31      80     21    924.0         0.262500          11.550000


In [52]:
supported_economics = (
    expected_clicks_by_card
    .merge(
        card_perf[
            [
                "card_id",
                "conversion_rate",
                "revenue_per_click"
            ]
        ],
        on="card_id",
        how="left"
    )
)

supported_economics["expected_sales"] = (
    supported_economics["expected_clicks"] *
    supported_economics["conversion_rate"]
)

supported_economics["expected_revenue"] = (
    supported_economics["expected_clicks"] *
    supported_economics["revenue_per_click"]
)

supported_economics = (
    supported_economics
    .sort_values("expected_revenue", ascending=False)
    .reset_index(drop=True)
)

print(supported_economics)

print()
print(
    "Expected replacement clicks:",
    supported_economics["expected_clicks"].sum()
)
print(
    "Expected replacement sales:",
    supported_economics["expected_sales"].sum()
)
print(
    "Expected replacement revenue:",
    supported_economics["expected_revenue"].sum()
)

    card_id  expected_clicks  conversion_rate  revenue_per_click  \
0        30       105.030075         0.316726          13.935943   
1        31        18.571429         0.262500          11.550000   
2         7         7.465897         0.351351          11.945946   
3        23         4.645161         0.326087          11.413043   
4         8         2.421245         0.440000          19.800000   
5        15         3.382576         0.469388          13.142857   
6        24         3.666667         0.328947          11.513158   
7        14         4.216532         0.241935           8.467742   
8         9         2.924051         0.352459          11.983607   
9         4         2.000000         0.357143          12.142857   
10       10         1.485981         0.360656          16.229508   
11       22         1.826087         0.379310          12.896552   
12       25         1.633256         0.333333          11.666667   
13        2         0.833333         0.476923   

In [53]:
# Attach historical support to every affected application's rank-1 set
affected_rank1_support = (
    rank1_sets
    .merge(
        historical_rank1_set_counts,
        on="rank1_cards",
        how="left"
    )
)

affected_rank1_support["historical_applications"] = (
    affected_rank1_support["historical_applications"]
    .fillna(0)
    .astype(int)
)

residual_apps = affected_rank1_support[
    affected_rank1_support["historical_applications"] < 10
].copy()

print("Residual applications:", len(residual_apps))

assert len(residual_apps) == 33

Residual applications: 33


In [54]:
rpc_map = (
    card_perf
    .set_index("card_id")["revenue_per_click"]
    .to_dict()
)

conversion_map = (
    card_perf
    .set_index("card_id")["conversion_rate"]
    .to_dict()
)


def mean_metric_for_rank1_set(cards, metric_map):
    values = [metric_map[card] for card in cards]
    return np.mean(values)


residual_apps["revenue_per_transfer_click"] = (
    residual_apps["rank1_cards"]
    .apply(
        lambda cards:
        mean_metric_for_rank1_set(cards, rpc_map)
    )
)

residual_apps["sales_per_transfer_click"] = (
    residual_apps["rank1_cards"]
    .apply(
        lambda cards:
        mean_metric_for_rank1_set(cards, conversion_map)
    )
)

residual_apps[
    [
        "application_id",
        "rank1_cards",
        "historical_applications",
        "revenue_per_transfer_click",
        "sales_per_transfer_click"
    ]
].head(20)

,application_id,rank1_cards,historical_applications,revenue_per_transfer_click,sales_per_transfer_click
3,K3E4cXh1QVNYRThaYjJaUHVoSjZydz090,"(31,)",4,11.550000,0.262500
22,MUMrTm1KV1NuYmRBOE9SRFFibTBCdz090,"(31,)",4,11.550000,0.262500
55,OUhQNDBBYmYzb2o3V3ZYV0I4anF1QT090,"(7, 10, 14)",0,12.214399,0.317981
58,Q0dlTVJRZ2I4aHNsYVdLeVBWOUgwdz090,"(9, 23)",8,11.698325,0.339273
60,Q3YrM2NUMHU3QnljVzVNNCtEbkdPUT090,"(9, 14, 15)",3,11.198069,0.354594
73,QlZrWVQxMGFLemcreFdDS2gvSXNjdz090,"(31,)",4,11.550000,0.262500
76,QzBxcnFBVWZ5TlRKanZEbjF5Vk1HUT090,"(10, 23)",7,13.821276,0.343371
79,R3I1VVBqZDNBWVIrK0l1RWVNdEladz090,"(10, 14)",3,12.348625,0.301296
91,RWszRlp3TVIvYWQ3ZmNwYmRicXFNdz090,"(31,)",4,11.550000,0.262500
114,TDRnRDM3L1NMNXh0VTZkNlQ1Q21sUT090,"(14, 22)",0,10.682147,0.310623


In [55]:
residual_full_transfer_revenue = (
    residual_apps["revenue_per_transfer_click"].sum()
)

residual_full_transfer_sales = (
    residual_apps["sales_per_transfer_click"].sum()
)

print(
    f"Residual applicants: {len(residual_apps)}"
)

print(
    f"Revenue if all 33 transfer once: "
    f"£{residual_full_transfer_revenue:,.2f}"
)

print(
    f"Expected sales if all 33 transfer once: "
    f"{residual_full_transfer_sales:.2f}"
)

print(
    f"Average revenue per transferred residual applicant: "
    f"£{residual_full_transfer_revenue / len(residual_apps):,.2f}"
)

Residual applicants: 33
Revenue if all 33 transfer once: £397.06
Expected sales if all 33 transfer once: 10.27
Average revenue per transferred residual applicant: £12.03


In [56]:
CARD29_TOTAL_REVENUE = 2772.00
CARD29_REVENUE_PER_SALE = 44.00

SUPPORTED_REPLACEMENT_REVENUE = supported_expected_revenue
RESIDUAL_FULL_TRANSFER_REVENUE = residual_full_transfer_revenue

sensitivity_rows = []

for retained_sales in range(0, 64):

    retained_revenue = (
        retained_sales * CARD29_REVENUE_PER_SALE
    )

    affected_card29_baseline = (
        CARD29_TOTAL_REVENUE - retained_revenue
    )

    revenue_gap_after_supported = (
        affected_card29_baseline -
        SUPPORTED_REPLACEMENT_REVENUE
    )

    required_residual_transfer = (
        revenue_gap_after_supported /
        RESIDUAL_FULL_TRANSFER_REVENUE
    )

    sensitivity_rows.append({
        "retained_card29_sales": retained_sales,
        "retained_card29_revenue": retained_revenue,
        "affected_card29_baseline_revenue": affected_card29_baseline,
        "supported_replacement_revenue": SUPPORTED_REPLACEMENT_REVENUE,
        "revenue_gap_after_supported": revenue_gap_after_supported,
        "required_residual_transfer_rate": required_residual_transfer
    })

break_even_sensitivity = pd.DataFrame(sensitivity_rows)

break_even_sensitivity.head()

,retained_card29_sales,retained_card29_revenue,affected_card29_baseline_revenue,supported_replacement_revenue,revenue_gap_after_supported,required_residual_transfer_rate
0,0,0.0,2772.0,2143.638511,628.361489,1.582526
1,1,44.0,2728.0,2143.638511,584.361489,1.471712
2,2,88.0,2684.0,2143.638511,540.361489,1.360898
3,3,132.0,2640.0,2143.638511,496.361489,1.250084
4,4,176.0,2596.0,2143.638511,452.361489,1.139270


In [57]:
selected_sales = [0, 5, 10, 13, 14, 15]

break_even_view = (
    break_even_sensitivity[
        break_even_sensitivity["retained_card29_sales"]
        .isin(selected_sales)
    ]
    .copy()
)

break_even_view[
    "required_residual_transfer_pct"
] = (
    break_even_view[
        "required_residual_transfer_rate"
    ] * 100
)

break_even_view[
    [
        "retained_card29_sales",
        "retained_card29_revenue",
        "affected_card29_baseline_revenue",
        "required_residual_transfer_pct"
    ]
]

,retained_card29_sales,retained_card29_revenue,affected_card29_baseline_revenue,required_residual_transfer_pct
0,0,0.0,2772.0,158.252556
5,5,220.0,2552.0,102.845656
10,10,440.0,2332.0,47.438756
13,13,572.0,2200.0,14.194616
14,14,616.0,2156.0,3.113235
15,15,660.0,2112.0,-7.968145


In [58]:
sales_needed_for_full_transfer_to_break_even = (
    CARD29_TOTAL_REVENUE
    - SUPPORTED_REPLACEMENT_REVENUE
    - RESIDUAL_FULL_TRANSFER_REVENUE
) / CARD29_REVENUE_PER_SALE

sales_needed_for_supported_only_to_break_even = (
    CARD29_TOTAL_REVENUE
    - SUPPORTED_REPLACEMENT_REVENUE
) / CARD29_REVENUE_PER_SALE

print(
    "Minimum retained sales for break-even to be achievable "
    "even with 100% residual transfer:",
    sales_needed_for_full_transfer_to_break_even
)

print(
    "Retained sales needed for the supported 282 alone "
    "to break even:",
    sales_needed_for_supported_only_to_break_even
)

Minimum retained sales for break-even to be achievable even with 100% residual transfer: 5.256796161354782
Retained sales needed for the supported 282 alone to break even: 14.280942940724096


# Conclusion

Card 29 should not be evaluated as a full-panel removal.

Of the 398 applicants who clicked Card 29:

- 83 (20.9%) had no alternative offer and should therefore retain Card 29;
- 315 (79.1%) had at least one alternative and form the relevant intervention population.

After suppressing Card 29 for these 315 applicants, likelihood ranks were recomputed using the
same competition-ranking method as the supplied data.

For 282 of the 315 affected applicants (89.5%), the resulting rank-1 card combination had at
least 10 historical observations in applications where Card 29 was absent. Historical behaviour
for these well-supported states was used to estimate click/no-click behaviour and redistribution
between the top-ranked cards.

These 282 applicants generate an estimated:

- 160.97 replacement clicks;
- 51.45 sales;
- £2,143.64 revenue.

The remaining 33 applicants have insufficient historical support and are therefore treated through
sensitivity analysis rather than point-estimated.

Because sales data are available only at card level, the number of Card 29 sales attributable to
the 83 retained sole-offer applicants cannot be observed directly. Break-even is therefore assessed
jointly across:

1. Card 29 sales/revenue retained among the 83 sole-offer applicants; and
2. transfer among the 33 residual applicants.

This supports testing conditional suppression/deprioritisation of Card 29 when alternatives are
available rather than permanent panel-wide removal.

In [59]:
OUTPUT_DIR = Path("../outputs/tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

intervention_summary.to_csv(
    OUTPUT_DIR / "conditional_intervention_summary.csv",
    index=False
)

alternative_distribution.to_csv(
    OUTPUT_DIR / "conditional_alternative_distribution.csv",
    index=False
)

rank1_distribution.to_csv(
    OUTPUT_DIR / "conditional_rank1_distribution.csv",
    index=False
)

expected_clicks_by_card.to_csv(
    OUTPUT_DIR / "conditional_expected_clicks_by_card.csv",
    index=False
)

supported_economics.to_csv(
    OUTPUT_DIR / "conditional_supported_economics.csv",
    index=False
)

break_even_sensitivity.to_csv(
    OUTPUT_DIR / "conditional_break_even_sensitivity.csv",
    index=False
)

residual_apps.to_csv(
    OUTPUT_DIR / "conditional_residual_apps.csv",
    index=False
)

print("Conditional counterfactual tables exported.")

Conditional counterfactual tables exported.
